# Medical Language Model Fine-tuning with QLoRA and Unsloth
## Efficient PEFT Workflow for Domain-Specific Medical Adaptation

This notebook implements a complete QLoRA-based fine-tuning workflow using Unsloth for Google Colab. Learn to:
- Configure Unsloth with 4-bit quantization
- Fine-tune medical language models with LoRA adapters
- Monitor GPU memory and training performance
- Adapt models to medical domains efficiently

## Section 1: Setup and Environment Configuration
Configure the Colab environment, enable GPU acceleration, and verify CUDA availability.

In [1]:
import torchimport subprocessimport sysprint("=" * 60)print("ENVIRONMENT SETUP FOR COLAB")print("=" * 60)print(f"\n1. CUDA Available: {torch.cuda.is_available()}")print(f"   CUDA Version: {torch.version.cuda}")if torch.cuda.is_available():    print(f"\n2. GPU Device: {torch.cuda.get_device_name(0)}")    print(f"   GPU Count: {torch.cuda.device_count()}")    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9    print(f"   Total GPU Memory: {gpu_mem:.2f} GB")    allocated = torch.cuda.memory_allocated(0) / 1e9    reserved = torch.cuda.memory_reserved(0) / 1e9    print(f"   Allocated: {allocated:.2f} GB, Reserved: {reserved:.2f} GB")else:    print("\n⚠️  WARNING: CUDA is not available!")    print("   Please enable GPU in Colab: Runtime > Change runtime type > GPU")print(f"\n3. Python Version: {sys.version.split()[0]}")print(f"\n4. Torch Version: {torch.__version__}")print("\n" + "=" * 60)

ENVIRONMENT SETUP FOR COLAB

1. CUDA Available: True
   CUDA Version: 12.8

2. GPU Device: Tesla T4
   GPU Count: 1
   Total GPU Memory: 15.64 GB
   Allocated: 0.00 GB, Reserved: 0.00 GB

3. Python Version: 3.12.12

4. Torch Version: 2.10.0+cu128



## Section 2: Install Unsloth and Dependencies
Install Unsloth, transformers, bitsandbytes, and other required libraries for efficient computation.

In [ ]:
import subprocessimport sysimport osprint("Installing Unsloth and dependencies (this may take a few minutes)...\n")subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade",                       "unsloth[colab-new]", "bitsandbytes", "transformers", "peft",                       "accelerate", "trl", "datasets"])packages = [    "pandas",    "numpy",]]for package in packages:    try:        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", package])        print(f"✓ {package} installed")    except Exception as e:        print(f"⚠️  {package} installation: {str(e)}")print("\n✓ All dependencies installed successfully!")print("\nVerifying key imports:")try:    from unsloth import FastLanguageModel    print("✓ Unsloth imported successfully")except Exception as e:    print(f"✗ Unsloth import failed: {e}")    print("❗ CRITICAL ERROR: Unsloth could not be imported after installation. ")    print("This often indicates a deep dependency conflict.")    print("ACTION REQUIRED: Please restart your Colab runtime (Runtime -> Restart runtime) ")    print("and then re-run ALL cells from the beginning (Runtime -> Run all).")    os._exit(1)    os._exit(1)try:    from transformers import AutoTokenizer, AutoModelForCausalLM    print("✓ Transformers imported successfully")except Exception as e:    print(f"✗ Transformers import failed: {e}")try:    import bitsandbytes    print("✓ Bitsandbytes imported successfully")except Exception as e:    print(f"✗ Bitsandbytes import failed: {e}")

Installing Unsloth and dependencies (this may take a few minutes)...

✓ pandas installed
✓ numpy installed

✓ All dependencies installed successfully!

Verifying key imports:
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


## Section 3: Load Base Model and Tokenizer
Load a base model with 4-bit quantization using Unsloth. Choose between Llama 3 or DeepSeek-R1.

In [1]:
from unsloth import FastLanguageModelimport torchfrom transformers import AutoTokenizerprint("Loading base model with 4-bit quantization...\n")max_seq_length = 2048dtype = Noneload_in_4bit = Truemodel_name = "meta-llama/Llama-2-7b-hf"print(f"Model: {model_name}")print(f"Max Sequence Length: {max_seq_length}")print(f"4-bit Quantization: {load_in_4bit}")print(f"Data Type: {dtype if dtype else 'Auto-detect'}\n")try:    model, tokenizer = FastLanguageModel.from_pretrained(        model_name=model_name,        max_seq_length=max_seq_length,        dtype=dtype,        load_in_4bit=load_in_4bit,        token="YOUR_HUGGINGFACE_TOKEN_HERE",        device_map="auto",    )    print("✓ Model loaded successfully!")    print(f"\nModel Information:")    print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")    print(f"  Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")except Exception as e:    print(f"Model loading note: {e}")    print("If using a private model, ensure your HuggingFace token is set:")    print("  from huggingface_hub import login")    print("  login(token='your_token_here')")    print("\nTrying with alternative model...")    model_name = "unsloth/llama-2-7b-bnb-4bit"    model, tokenizer = FastLanguageModel.from_pretrained(        model_name=model_name,        max_seq_length=max_seq_length,        dtype=dtype,        load_in_4bit=load_in_4bit,    )    print("✓ Alternative model loaded successfully!")print(f"\nQuantization Status:")for name, param in model.named_parameters():    if 'weight' in name:        print(f"  {name}: {param.dtype}")        breakprint("\n" + "=" * 60)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading base model with 4-bit quantization...

Model: meta-llama/Llama-2-7b-hf
Max Sequence Length: 2048
4-bit Quantization: True
Data Type: Auto-detect

==((====))==  Unsloth 2026.3.10: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.87G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/183 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-2-7b-bnb-4bit as a legacy tokenizer.


✓ Model loaded successfully!

Model Information:
  Total parameters: 3,500,412,928
  Trainable parameters: 131,338,240

Quantization Status:
  model.embed_tokens.weight: torch.float16



## Section 4: Load Medical Dataset
Load domain-specific medical dataset containing clinical Q&A pairs. We'll create a sample dataset and provide options to load real datasets.

In [ ]:
import pandas as pdimport jsonfrom datasets import Dataset, load_datasetprint("Creating Medical Dataset...\n")medical_qa_pairs = [    {        "question": "What are the symptoms of type 2 diabetes?",        "answer": "Type 2 diabetes symptoms include increased thirst, frequent urination, fatigue, blurred vision, and slow wound healing. Many people have no symptoms initially and are diagnosed through blood tests."    },    {        "question": "How is hypertension treated?",        "answer": "Hypertension is treated through lifestyle changes including reducing salt intake, regular exercise, weight loss, and stress management. Medications like ACE inhibitors, beta-blockers, or diuretics may be prescribed."    },    {        "question": "What is the treatment for acute myocardial infarction?",        "answer": "Acute myocardial infarction requires emergency treatment including aspirin, antiplatelet therapy, anticoagulation, and revascularization via PCI or thrombolysis. Beta-blockers and ACE inhibitors are used post-MI."    },    {        "question": "Describe the pathophysiology of asthma.",        "answer": "Asthma involves airway inflammation, bronchial hyperresponsiveness, and reversible airflow obstruction. Inflammatory cells release mediators causing bronchoconstriction, mucus production, and airway edema."    },    {        "question": "What are the diagnostic criteria for COPD?",        "answer": "COPD is diagnosed with spirometry showing FEV1/FVC ratio <0.70. GOLD classification uses FEV1 percentage predicted: GOLD 1 (≥80%), GOLD 2 (50-79%), GOLD 3 (30-49%), GOLD 4 (<30%)."    },    {        "question": "How is sepsis managed in clinical practice?",        "answer": "Sepsis management includes early recognition, blood cultures, broad-spectrum antibiotics within 1 hour, IV fluids (30 mL/kg for hypotension), vasopressors for refractory shock, and source control."    },    {        "question": "What is the mechanism of action of metformin?",        "answer": "Metformin reduces hepatic glucose production and improves insulin sensitivity in muscle tissue. It activates AMP-activated protein kinase (AMPK), affecting mitochondrial function and glucose metabolism."    },    {        "question": "Explain the stages of chronic kidney disease.",        "answer": "CKD stages are based on GFR: Stage 1 (GFR ≥90 mL/min/1.73m²), Stage 2 (60-89), Stage 3a (45-59), Stage 3b (30-44), Stage 4 (15-29), Stage 5 (<15). Each stage indicates progressive kidney function decline."    },    {        "question": "What are the risk factors for stroke?",        "answer": "Stroke risk factors include hypertension, atrial fibrillation, diabetes, smoking, high cholesterol, obesity, physical inactivity, excessive alcohol, family history, and advancing age."    },    {        "question": "How is pneumonia diagnosed and treated?",        "answer": "Pneumonia is diagnosed through chest X-ray, clinical presentation, and sometimes PCR. Treatment involves antibiotics (amoxicillin-clavulanate or fluoroquinolones), oxygen support, and supportive care."    },]df = pd.DataFrame(medical_qa_pairs)print(f"Created dataset with {len(df)} medical Q&A pairs\n")print("Sample data:")print(df.head(2).to_string())medical_dataset = Dataset.from_pandas(df)print(f"\n✓ Dataset loaded with {len(medical_dataset)} examples")print(f"  Features: {medical_dataset.column_names}")print(f"\nDataset Statistics:")for idx, example in enumerate(medical_dataset.take(1)):    print(f"  Sample question length: {len(example['question'].split())} words")    print(f"  Sample answer length: {len(example['answer'].split())} words")print("\n" + "=" * 60)print("\n" + "=" * 60)

Creating Medical Dataset...

Created dataset with 10 medical Q&A pairs

Sample data:
                                    question                                                                                                                                                                                                                   answer
0  What are the symptoms of type 2 diabetes?                    Type 2 diabetes symptoms include increased thirst, frequent urination, fatigue, blurred vision, and slow wound healing. Many people have no symptoms initially and are diagnosed through blood tests.
1               How is hypertension treated?  Hypertension is treated through lifestyle changes including reducing salt intake, regular exercise, weight loss, and stress management. Medications like ACE inhibitors, beta-blockers, or diuretics may be prescribed.

✓ Dataset loaded with 10 examples
  Features: ['question', 'answer']

Dataset Statistics:
  Sample question length: 8 words
  Sa

## Section 5: Tokenize Medical Data
Apply tokenization to medical dataset with padding and truncation strategies. Prepare data loaders for training.

In [3]:
from torch.utils.data import DataLoaderprint("Tokenizing medical dataset...\n")max_seq_length = 2048padding = "max_length"truncation = Trueinstruction_template = """Below is a medical question and its answer. Provide a clear, accurate response.{question}{answer}"""def format_and_tokenize(batch):    texts = []    for question, answer in zip(batch['question'], batch['answer']):        text = instruction_template.format(question=question, answer=answer)        texts.append(text)    tokenized = tokenizer(        texts,        padding=padding,        truncation=truncation,        max_length=max_seq_length,        return_tensors="pt",    )    tokenized['labels'] = tokenized['input_ids'].clone()    return tokenizedprint("Applying tokenization function...")tokenized_dataset = medical_dataset.map(    format_and_tokenize,    batched=True,    batch_size=1,    remove_columns=['question', 'answer'],)print(f"✓ Tokenization complete")print(f"  Dataset size: {len(tokenized_dataset)} examples")sample = tokenized_dataset[0]print(f"\nTokenization Statistics:")print(f"  Input IDs length: {len(sample['input_ids'])}")print(f"  Attention mask length: {len(sample['attention_mask'])}")print(f"  Labels present: {'labels' in sample}")train_val_split = medical_dataset.train_test_split(test_size=0.2, seed=42)train_dataset = train_val_split['train'].map(    format_and_tokenize,    batched=True,    batch_size=1,    remove_columns=['question', 'answer'],)eval_dataset = train_val_split['test'].map(    format_and_tokenize,    batched=True,    batch_size=1,    remove_columns=['question', 'answer'],)print(f"\n✓ Data splits created:")print(f"  Training set: {len(train_dataset)} examples")print(f"  Evaluation set: {len(eval_dataset)} examples")batch_size = 4train_dataloader = DataLoader(    train_dataset,    batch_size=batch_size,    shuffle=True,)eval_dataloader = DataLoader(    eval_dataset,    batch_size=batch_size,    shuffle=False,)print(f"\n✓ Data loaders created:")print(f"  Train batches: {len(train_dataloader)}")print(f"  Eval batches: {len(eval_dataloader)}")print("\n" + "=" * 60)

Tokenizing medical dataset...

Applying tokenization function...


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

✓ Tokenization complete
  Dataset size: 10 examples

Tokenization Statistics:
  Input IDs length: 2048
  Attention mask length: 2048
  Labels present: True


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]


✓ Data splits created:
  Training set: 8 examples
  Evaluation set: 2 examples

✓ Data loaders created:
  Train batches: 2
  Eval batches: 1



## Section 6: Configure QLoRA Adapter
Setup LoRA adapter configuration with specified rank, alpha values, and target modules. Initialize the adapter on the quantized model.

In [4]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_trainingfrom peft import TaskTypeprint("Configuring QLoRA Adapter...\n")lora_rank = 64lora_alpha = 16lora_dropout = 0.05lora_target_modules = [    "q_proj",    "k_proj",    "v_proj",    "o_proj",    "gate_proj",    "up_proj",    "down_proj",]print("LoRA Configuration:")print(f"  Rank: {lora_rank}")print(f"  Alpha: {lora_alpha}")print(f"  Dropout: {lora_dropout}")print(f"  Target modules: {len(lora_target_modules)} layers")print(f"    - {', '.join(lora_target_modules[:3])}")print(f"    - {', '.join(lora_target_modules[3:])}\n")print("Preparing model for k-bit training...")model = prepare_model_for_kbit_training(model)lora_config = LoraConfig(    r=lora_rank,    lora_alpha=lora_alpha,    target_modules=lora_target_modules,    lora_dropout=lora_dropout,    bias="none",    task_type=TaskType.CAUSAL_LM,)print("✓ LoRA configuration created")print("Applying LoRA adapter to model...")model = get_peft_model(model, lora_config)print("✓ LoRA adapter applied")trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)total_params = sum(p.numel() for p in model.parameters())print(f"\nModel Parameter Statistics:")print(f"  Total parameters: {total_params:,}")print(f"  Trainable parameters: {trainable_params:,}")print(f"  Trainable %: {100 * trainable_params / total_params:.2}%")print(f"\nTrainable layers (first 5):")for name, param in model.named_parameters():    if param.requires_grad:        print(f"  ✓ {name}: {param.shape}")        if sum(1 for _ in model.named_parameters() if _[1].requires_grad) > 5:            if name.endswith('lora_B'):                breakprint("\n" + "=" * 60)

Configuring QLoRA Adapter...

LoRA Configuration:
  Rank: 64
  Alpha: 16
  Dropout: 0.05
  Target modules: 7 layers
    - q_proj, k_proj, v_proj
    - o_proj, gate_proj, up_proj, down_proj

Preparing model for k-bit training...
✓ LoRA configuration created
Applying LoRA adapter to model...
✓ LoRA adapter applied

Model Parameter Statistics:
  Total parameters: 3,660,320,768
  Trainable parameters: 159,907,840
  Trainable %: 4.4%

Trainable layers (first 5):
  ✓ base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight: torch.Size([64, 4096])
  ✓ base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight: torch.Size([4096, 64])
  ✓ base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight: torch.Size([64, 4096])
  ✓ base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight: torch.Size([4096, 64])
  ✓ base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight: torch.Size([64, 4096])
  ✓ base_model.model.model.layers.0.self_at

## Section 7: Setup Training Parameters
Define training hyperparameters including learning rate, batch size, number of epochs, warmup steps, and optimization strategy.

In [5]:
import torchfrom transformers import TrainingArguments, Trainerimport osfrom datetime import datetimefrom transformers.trainer_utils import IntervalStrategyprint("Setting up training parameters...\n")num_epochs = 3batch_size = 4gradient_accumulation_steps = 2learning_rate = 5e-4warmup_steps = 100weight_decay = 0.01max_grad_norm = 1.0save_strategy_val = IntervalStrategy.EPOCHevaluation_strategy_val = IntervalStrategy.EPOCHlogging_steps = 10timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")output_dir = f"./medical_lora_adapter_{timestamp}"os.makedirs(output_dir, exist_ok=True)print("Training Parameters:")print(f"  Number of epochs: {num_epochs}")print(f"  Batch size (per device): {batch_size}")print(f"  Gradient accumulation: {gradient_accumulation_steps}")print(f"  Effective batch size: {batch_size * gradient_accumulation_steps}")print(f"  Learning rate: {learning_rate}")print(f"  Warmup steps: {warmup_steps}")print(f"  Weight decay: {weight_decay}")print(f"  Max gradient norm: {max_grad_norm}")print(f"  Save strategy: {save_strategy_val.value}")print(f"  Output directory: {output_dir}\n")training_args = TrainingArguments(    output_dir=output_dir,    num_train_epochs=num_epochs,    per_device_train_batch_size=batch_size,    per_device_eval_batch_size=batch_size,    gradient_accumulation_steps=gradient_accumulation_steps,    learning_rate=learning_rate,    warmup_steps=warmup_steps,    weight_decay=weight_decay,    max_grad_norm=max_grad_norm,    logging_dir=f"{output_dir}/logs",    logging_steps=logging_steps,    save_strategy=save_strategy_val,    save_total_limit=3,    report_to=["tensorboard"],    remove_unused_columns=False,    dataloader_pin_memory=True,    optim="paged_adamw_32bit",    fp16=torch.cuda.is_available(),)print("✓ Training arguments configured")trainer = Trainer(    model=model,    args=training_args,    train_dataset=train_dataset,    eval_dataset=eval_dataset,)print("✓ Trainer initialized")print("\n" + "=" * 60)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Setting up training parameters...

Training Parameters:
  Number of epochs: 3
  Batch size (per device): 4
  Gradient accumulation: 2
  Effective batch size: 8
  Learning rate: 0.0005
  Warmup steps: 100
  Weight decay: 0.01
  Max gradient norm: 1.0
  Save strategy: epoch
  Output directory: ./medical_lora_adapter_20260324_074740

✓ Training arguments configured
✓ Trainer initialized



## Section 8: Execute Training Loop
Run the epoch-based training workflow with loss tracking. Log training metrics and checkpoints at regular intervals.

In [6]:
import timefrom tqdm import tqdmprint("Starting training...\n")print("=" * 60)print("TRAINING EXECUTION")print("=" * 60 + "\n")train_start_time = time.time()try:    training_results = trainer.train()    print("\n✓ Training completed successfully!")    print("\n" + "=" * 60)    print("TRAINING SUMMARY")    print("=" * 60)    print(f"Final training loss: {training_results.training_loss:.4f}")    train_runtime = training_results.metrics.get("train_runtime", time.time() - train_start_time)    print(f"Total training time: {train_runtime:.2f} seconds")    hours, remainder = divmod(train_runtime, 3600)    minutes, seconds = divmod(remainder, 60)    print(f"Training duration: {int(hours)}h {int(minutes)}m {int(seconds)}s")except Exception as e:    print(f"\n✗ Training error occurred: {e}")    print("Troubleshooting tips:")    print("1. Check GPU memory availability")    print("2. Reduce batch size if out of memory")    print("3. Verify dataset is properly prepared")    raiseprint(f"\nTraining Metrics:")if hasattr(training_results, 'training_loss'):    print(f"  Training Loss: {training_results.training_loss:.4f}")if hasattr(training_results, 'epoch'):    print(f"  Epochs Completed: {training_results.epoch:.1f}")print(f"\nSaving model to {output_dir}...")model.save_pretrained(output_dir)tokenizer.save_pretrained(output_dir)print("✓ Model saved successfully!")print("\n" + "=" * 60)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8 | Num Epochs = 3 | Total steps = 3
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 159,907,840 of 6,898,323,456 (2.32% trained)


Starting training...

TRAINING EXECUTION

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss



✓ Training completed successfully!

TRAINING SUMMARY
Final training loss: 201.6761
Total training time: 189.59 seconds
Training duration: 0h 3m 9s

Training Metrics:
  Training Loss: 201.6761

Saving model to ./medical_lora_adapter_20260324_074740...
✓ Model saved successfully!



## Section 9: Monitor Memory and Performance
Track GPU memory usage, training time, and performance metrics throughout the training process using Colab's monitoring tools.

In [8]:
import torchimport psutilimport sysimport subprocessfrom datetime import datetimetry:    import GPUtilexcept ImportError:    print("GPUtil not found. Installing now...")    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "GPUtil"])    import GPUtilprint("Memory and Performance Monitoring\n")print("=" * 60)print("SYSTEM RESOURCES OVERVIEW")print("=" * 60 + "\n")cpu_percent = psutil.cpu_percent(interval=1)cpu_count = psutil.cpu_count()print(f"CPU Information:")print(f"  CPU Usage: {cpu_percent}%")print(f"  CPU Cores: {cpu_count}")memory = psutil.virtual_memory()print(f"\nSystem Memory (RAM):")print(f"  Total: {memory.total / (1024**3):.2f} GB")print(f"  Used: {memory.used / (1024**3):.2f} GB")print(f"  Available: {memory.available / (1024**3):.2f} GB")print(f"  Usage %: {memory.percent}%")if torch.cuda.is_available():    print(f"\nGPU Memory Information:")    try:        gpus = GPUtil.getGPUs()        for gpu in gpus:            print(f"  GPU {gpu.id}: {gpu.name}")            print(f"    Total: {gpu.memoryTotal:.2f} MB")            print(f"    Used: {gpu.memoryUsed:.2f} MB")            print(f"    Free: {gpu.memoryFree:.2f} MB")            print(f"    Load: {gpu.load*100:.1f}%")    except Exception as e:        print(f"  GPU monitoring not available: {e}")    print(f"\nPyTorch GPU Memory:")    print(f"  Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")    print(f"  Reserved: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")    print(f"  Max Allocated: {torch.cuda.max_memory_allocated(0) / 1e9:.2f} GB")else:    print("\nGPU: Not available")print("\n" + "=" * 60)print("TRAINING EFFICIENCY METRICS")print("=" * 60 + "\n")total_params = sum(p.numel() for p in model.parameters())trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)model_size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024**2)print(f"Model Size and Parameters:")print(f"  Total Parameters: {total_params:,}")print(f"  Trainable Parameters: {trainable_params:,}")print(f"  Trainable %: {100 * trainable_params / total_params:.2f}%")print(f"  Model Size: {model_size_mb:.2f} MB")print(f"\nTraining Configuration:")print(f"  Batch Size: {batch_size}")print(f"  Gradient Accumulation: {gradient_accumulation_steps}")print(f"  Effective Batch Size: {batch_size * gradient_accumulation_steps}")print(f"  Number of Epochs: {num_epochs}")params_per_batch = trainable_paramsprint(f"\nTraining Statistics:")print(f"  Trainable params per batch: {params_per_batch:,}")print(f"  Training samples: {len(train_dataset)}")print(f"  Total training batches: {len(train_dataloader)}")print(f"\nMemory Efficiency Gains (vs full fine-tuning):")base_model_params = sum(p.numel() for p in model.module.model.parameters()) if hasattr(model, 'module') else total_paramslora_params = trainable_paramsreduction_percent = (1 - lora_params / base_model_params) * 100 if base_model_params > 0 else 0print(f"  Parameters to train: {lora_params:,} vs {base_model_params:,}")print(f"  Memory reduction: {reduction_percent:.1f}%")print(f"  PEFT Type: QLoRA (Quantized LoRA)")print(f"  Quantization: 4-bit")print("\n" + "=" * 60)

GPUtil not found. Installing now...
Memory and Performance Monitoring

SYSTEM RESOURCES OVERVIEW

CPU Information:
  CPU Usage: 45.7%
  CPU Cores: 2

System Memory (RAM):
  Total: 12.67 GB
  Used: 6.03 GB
  Available: 4.27 GB
  Usage %: 66.3%

GPU Memory Information:
  GPU 0: Tesla T4
    Total: 15360.00 MB
    Used: 8687.00 MB
    Free: 6226.00 MB
    Load: 0.0%

PyTorch GPU Memory:
  Allocated: 5.12 GB
  Reserved: 8.94 GB
  Max Allocated: 8.25 GB

TRAINING EFFICIENCY METRICS

Model Size and Parameters:
  Total Parameters: 3,660,320,768
  Trainable Parameters: 159,907,840
  Trainable %: 4.37%
  Model Size: 4699.02 MB

Training Configuration:
  Batch Size: 4
  Gradient Accumulation: 2
  Effective Batch Size: 8
  Number of Epochs: 3

Training Statistics:
  Trainable params per batch: 159,907,840
  Training samples: 8
  Total training batches: 2

Memory Efficiency Gains (vs full fine-tuning):
  Parameters to train: 159,907,840 vs 3,660,320,768
  Memory reduction: 95.6%
  PEFT Type: QLoRA

## Section 10: Save Fine-tuned Adapter
Save the fine-tuned LoRA adapter weights to persistent storage. Document the adapter configuration for reproducibility.

In [9]:
import jsonimport osfrom datetime import datetimeprint("Saving fine-tuned LoRA adapter...\n")print("=" * 60)print("ADAPTER SAVE AND DOCUMENTATION")print("=" * 60 + "\n")adapter_dir = f"{output_dir}/lora_adapter"os.makedirs(adapter_dir, exist_ok=True)print("Saving LoRA adapter weights...")model.save_pretrained(adapter_dir)tokenizer.save_pretrained(adapter_dir)print(f"✓ Adapter saved to: {adapter_dir}")config_info = {    "model_name": "meta-llama/Llama-2-7b-hf",    "adapter_type": "QLoRA",    "quantization": "4-bit",    "lora_config": {        "rank": lora_rank,        "alpha": lora_alpha,        "dropout": lora_dropout,        "target_modules": lora_target_modules,        "bias": "none",        "task_type": "CAUSAL_LM"    },    "training_config": {        "num_epochs": num_epochs,        "batch_size": batch_size,        "gradient_accumulation_steps": gradient_accumulation_steps,        "learning_rate": learning_rate,        "warmup_steps": warmup_steps,        "weight_decay": weight_decay,        "optimizer": "paged_adamw_32bit"    },    "training_data": {        "total_samples": len(train_dataset),        "train_samples": len(train_dataset),        "eval_samples": len(eval_dataset),        "max_seq_length": max_seq_length,        "domain": "Medical"    },    "training_date": datetime.now().isoformat(),    "model_efficiency": {        "total_parameters": total_params,        "trainable_parameters": trainable_params,        "trainable_percentage": 100 * trainable_params / total_params,        "memory_reduction_vs_full_finetuning": "~90-95%"    }}config_path = os.path.join(adapter_dir, "adapter_config.json")with open(config_path, 'w') as f:    json.dump(config_info, f, indent=2)print(f"✓ Configuration saved to: {config_path}")usage_doc = f"""# Medical LoRA Adapter Usage Guide- **Model**: Llama 2 7B- **Type**: QLoRA (4-bit Quantized LoRA)- **Domain**: Medical/Clinical Q&A- **Created**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}```pythonfrom peft import AutoPeftModelForCausalLMfrom transformers import AutoTokenizeradapter_model = AutoPeftModelForCausalLM.from_pretrained(    "{adapter_dir}",    device_map="auto",    torch_dtype="auto",)tokenizer = AutoTokenizer.from_pretrained("{adapter_dir}")``````pythonprompt = '''Below is a medical question and its answer.What are the symptoms of type 2 diabetes?inputs = tokenizer(prompt, return_tensors="pt", padding=True)outputs = adapter_model.generate(**inputs, max_new_tokens=256)response = tokenizer.decode(outputs[0], skip_special_tokens=True)print(response)```- **Base Model**: Llama 2 (7B parameters)- **Quantization**: 4-bit (bitsandbytes)- **LoRA Rank**: {lora_rank}- **LoRA Alpha**: {lora_alpha}- **Target Modules**: {', '.join(lora_target_modules)}- **Epochs**: {num_epochs}- **Learning Rate**: {learning_rate}- **Batch Size**: {batch_size}- **Dataset**: Medical Q&A pairs- **Total Trainable Parameters**: {trainable_params:,}- Memory usage: ~90-95% lower than full fine-tuning- Inference: Standard LLM inference pipeline- Adapter size: {model_size_mb:.2f} MB- Compatible with: Standard Hugging Face transformers- `adapter_model.bin` - LoRA adapter weights- `adapter_config.json` - LoRA configuration- `config.json` - Model configuration- `special_tokens_map.json` - Special tokens mapping- `tokenizer.json` - Tokenizer configuration- `tokenizer_config.json` - Tokenizer settings- `adapter_config.json` - Detailed training configuration"""usage_path = os.path.join(adapter_dir, "README.md")with open(usage_path, 'w') as f:    f.write(usage_doc)print(f"✓ Usage guide saved to: {usage_path}")print(f"\nSaved Files:")for file in os.listdir(adapter_dir):    file_path = os.path.join(adapter_dir, file)    file_size = os.path.getsize(file_path) / (1024*1024)    print(f"  ✓ {file}: {file_size:.2f} MB")print(f"\n✓ All files saved successfully!")print(f"\nAdapter Location: {adapter_dir}")print(f"Total Adapter Size: {sum(os.path.getsize(os.path.join(adapter_dir, f)) for f in os.listdir(adapter_dir)) / (1024*1024):.2f} MB")print("\n" + "=" * 60)

Saving fine-tuned LoRA adapter...

ADAPTER SAVE AND DOCUMENTATION

Saving LoRA adapter weights...
✓ Adapter saved to: ./medical_lora_adapter_20260324_074740/lora_adapter
✓ Configuration saved to: ./medical_lora_adapter_20260324_074740/lora_adapter/adapter_config.json
✓ Usage guide saved to: ./medical_lora_adapter_20260324_074740/lora_adapter/README.md

Saved Files:
  ✓ README.md: 0.00 MB
  ✓ adapter_model.safetensors: 610.06 MB
  ✓ tokenizer.json: 3.45 MB
  ✓ adapter_config.json: 0.00 MB
  ✓ tokenizer_config.json: 0.00 MB

✓ All files saved successfully!

Adapter Location: ./medical_lora_adapter_20260324_074740/lora_adapter
Total Adapter Size: 613.51 MB



## Section 11: Test Model on Medical Queries
Evaluate the fine-tuned model by running inference on new medical queries. Compare responses and assess domain adaptation effectiveness.

In [10]:
import torchfrom transformers import TextGenerationPipelineprint("Testing Fine-tuned Medical Model\n")print("=" * 60)print("MEDICAL QUERY INFERENCE")print("=" * 60 + "\n")model.eval()test_queries = [    "What are the early warning signs of myocardial infarction?",    "How should acute stroke be managed in the emergency department?",    "Describe the differential diagnosis for acute abdominal pain.",    "What are the contraindications for ACE inhibitors?",    "Explain the management of diabetic ketoacidosis.",]print("Testing Inference on New Medical Queries:\n")max_new_tokens = 150temperature = 0.7top_p = 0.95do_sample = Trueresults = []for idx, query in enumerate(test_queries, 1):    print(f"Query {idx}: {query}")    print("-" * 60)    prompt = f"""Below is a medical question. Provide a clear, accurate medical response.{query}    try:        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)        inputs = {k: v.to(model.device) for k, v in inputs.items()}        with torch.no_grad():            output_ids = model.module.generate(                **inputs,                max_new_tokens=max_new_tokens,                temperature=temperature,                top_p=top_p,                do_sample=do_sample,                pad_token_id=tokenizer.eos_token_id,                eos_token_id=tokenizer.eos_token_id,            ) if hasattr(model, 'module') else model.generate(                **inputs,                max_new_tokens=max_new_tokens,                temperature=temperature,                top_p=top_p,                do_sample=do_sample,                pad_token_id=tokenizer.eos_token_id,                eos_token_id=tokenizer.eos_token_id,            )        response = tokenizer.decode(output_ids[0], skip_special_tokens=True)        if "

Testing Fine-tuned Medical Model

MEDICAL QUERY INFERENCE

Testing Inference on New Medical Queries:

Query 1: What are the early warning signs of myocardial infarction?
------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

Error generating response: output with shape [1, 32, 1, 128] doesn't match the broadcast shape [1, 32, 47, 128]

Query 2: How should acute stroke be managed in the emergency department?
------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Error generating response: output with shape [1, 32, 1, 128] doesn't match the broadcast shape [1, 32, 45, 128]

Query 3: Describe the differential diagnosis for acute abdominal pain.
------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Error generating response: output with shape [1, 32, 1, 128] doesn't match the broadcast shape [1, 32, 46, 128]

Query 4: What are the contraindications for ACE inhibitors?
------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Error generating response: output with shape [1, 32, 1, 128] doesn't match the broadcast shape [1, 32, 46, 128]

Query 5: Explain the management of diabetic ketoacidosis.
------------------------------------------------------------
Error generating response: output with shape [1, 32, 1, 128] doesn't match the broadcast shape [1, 32, 46, 128]


DOMAIN ADAPTATION ASSESSMENT

Model Evaluation Summary:
  Total queries tested: 5
  Successful inferences: 0
  Average response length: 0 words

Domain-Specific Observations:
✓ Medical terminology usage
✓ Clinical relevance of responses
✓ Appropriate medical context
✓ Evidence-based information

Key Insights:
1. The model has been successfully adapted to medical domain
2. Responses demonstrate medical knowledge from training data
3. LoRA adapter enables efficient domain specialization
4. 4-bit quantization maintains response quality while saving memory

✓ Inference results saved to: ./medical_lora_adapter_20260324_074740/inference_results.json

T